# Tier-C — Statistical rigour (multi-seed + significance + calibration)

Phase-2 Tier C turns the single-seed Tier-A/B *findings* into statistically defensible *claims*. It
**reuses the dual-GPU sharded runner** (each shard pinned to one T4 via `CUDA_VISIBLE_DEVICES`) and the
**per-run S3 resume** (every finished run pushes its CSV + checkpoint, so a session time-out costs at most the
run in flight — relaunch and it skips everything done).

Three things this notebook adds on top of training:
1. **Multi-seed aggregation** — mean ± std and a 95% CI across seeds, per (model, symbol, horizon).
2. **Significance tests** — paired tests between model pairs (across the seed×symbol×horizon cells) *and*
   a per-sample paired bootstrap + McNemar at a representative cell (reloading checkpoints from S3).
3. **Probability calibration** — temperature scaling, reporting Expected Calibration Error (ECE) before/after.

Headline pairs we want significance on:
- **ConvMambaLOB vs MambaLOB-base** (does the Tier-B conv front-end *really* help?)
- **MambaLOB(all) vs MambaLOB(base40)** (does Tier-A microstructure feature engineering *really* help?)
- **MambaLOB / ConvMambaLOB vs TLOB** (quantify the gap to the transformer with uncertainty)

Setup: enable **GPU T4 x2**; secrets `GH_PAT`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`.
Clean S3 layout: `experiments/tier_c/{results,checkpoints,figures}/`.


## 1. Runtime check (expect 2 GPUs)

In [ ]:
import torch, platform
print("Python:", platform.python_version(), "| Torch:", torch.__version__)
n = torch.cuda.device_count()
print(f"GPUs visible: {n}")
for i in range(n):
    print(f"  cuda:{i} = {torch.cuda.get_device_name(i)}")
assert n >= 1, "Enable GPU. For 2x speedup choose 'GPU T4 x2'."
if n < 2:
    print("Only 1 GPU -> will run a single shard (still works, no speedup).")

## 2. Get the project code (GH_PAT secret)

In [ ]:
import sys, subprocess, pathlib
REPO_URL = "https://github.com/rajjoseph48/nse-lob-capstone.git"
REPO_DIR = "nse-lob-capstone"
def _get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v
    except Exception:
        pass
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    import os
    return os.environ.get(name, "")
GITHUB_TOKEN = _get_secret("GH_PAT")
def add_modeling(base):
    base = pathlib.Path(base)
    for cand in (base / "modeling", base):
        if (cand / "models.py").exists():
            sys.path.insert(0, str(cand.resolve()))
            return str(cand.resolve())
    return None
MODELING_DIR = None
for c in (".", REPO_DIR, "/kaggle/working/" + REPO_DIR, "/content/" + REPO_DIR):
    MODELING_DIR = add_modeling(c)
    if MODELING_DIR:
        break
if not MODELING_DIR:
    url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", url], check=True)
    MODELING_DIR = add_modeling(REPO_DIR)
print("modeling/ dir:", MODELING_DIR)

## 3. Install deps (Mamba kernel + boto3 + scipy for the tests)

In [ ]:
import subprocess, sys
def pipq(*pkgs): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
pipq("boto3", "scipy")
pipq("ninja", "packaging", "setuptools", "wheel")
pipq("--no-build-isolation", "causal-conv1d")
pipq("--no-build-isolation", "mamba-ssm")
try:
    import mamba_ssm
    print("install step done | mamba-ssm", mamba_ssm.__version__)
except Exception as e:
    print("install step done | mamba-ssm NOT importable -> pure-PyTorch fallback:", repr(e))

## 4. Download NSE data from S3 (shared by both shards)

In [ ]:
import os, boto3, pathlib
os.environ["AWS_ACCESS_KEY_ID"] = _get_secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = _get_secret("AWS_SECRET_ACCESS_KEY")
BUCKET, PREFIX, REGION = "lob-capstone-data", "lob-data/dhan/", "ap-south-2"
DATA_DIR_NSE = "nse_data/dhan"; pathlib.Path(DATA_DIR_NSE).mkdir(parents=True, exist_ok=True)
s3 = boto3.client("s3", region_name=REGION)
n = 0
for o in s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX).get("Contents", []):
    if o["Size"] == 0:
        continue
    dst = os.path.join(DATA_DIR_NSE, o["Key"].split("/")[-1])
    if not os.path.exists(dst):
        s3.download_file(BUCKET, o["Key"], dst)
    n += 1
print(f"{n} parquet files in {DATA_DIR_NSE}")

## 5. Build the multi-seed run-list + resume
We multi-seed only the **headline configurations** (running every model x feature x seed would be wasteful).
`SEEDS` is the knob — default 3 seeds; bump to 5 for tighter CIs. Note **TLOB dominates runtime** (~1 h/run vs
minutes for Mamba), so the grid below is ~3-6 h on 2xT4 across one or two sessions; resume makes that safe.

In [ ]:
import json, pathlib, pandas as pd
from nbenv import s3_client, s3_pull_area

AREA = "tier_c"
MERGED = pathlib.Path("results/tier_c.csv")
SEEDS = [0, 1, 2]                                    # bump to [0,1,2,3,4] for 5-seed CIs

# Headline configs: (model, feature_set). 'all' = best from Tier A.
CONFIGS = [
    ("mambalob",     "base40"),                      # Tier-A reference (raw LOB)
    ("mambalob",     "all"),                         # Tier-A treatment (+microstructure)
    ("convmambalob", "all"),                         # Tier-B best variant
    ("tlob",         "all"),                         # transformer SOTA target
]
SYMBOLS = ["NIFTY", "BANKNIFTY"]
HORIZONS = [10, 100]

GRID = []
for (model, fs) in CONFIGS:
    for sym in SYMBOLS:
        for h in HORIZONS:
            for sd in SEEDS:
                GRID.append({"model": model, "symbol": sym, "feature_set": fs,
                             "horizon": h, "label_scheme": "A", "seed": sd})

# Resume: pull merged + each shard CSV; skip anything already done (per-run granularity).
_s3 = s3_client(); pathlib.Path("results").mkdir(exist_ok=True)
s3_pull_area(_s3, MERGED, AREA)
shard_csvs = [pathlib.Path(f"results/shard{i}.csv") for i in range(2)]
for sc in shard_csvs:
    s3_pull_area(_s3, sc, AREA)
done = set()
for f in [MERGED, *shard_csvs]:
    if f.exists():
        d = pd.read_csv(f)
        done |= {(r.model, r.symbol, r.feature_set, int(r.horizon), r.label_scheme, int(r.seed))
                 for r in d.itertuples()}
def _key(s):
    return (s["model"], s["symbol"], s["feature_set"], int(s["horizon"]), s["label_scheme"], int(s["seed"]))
todo = [s for s in GRID if _key(s) not in done]
pathlib.Path("runlist.json").write_text(json.dumps(todo))
print(f"grid={len(GRID)} | done={len(done)} | to run this session={len(todo)}")
print(f"(TLOB runs in to-do: {sum(1 for s in todo if s['model']=='tlob')} -- these are the slow ones)")
todo[:8]

## 6. Launch one shard per GPU (concurrent) and stream logs
Each shard is a subprocess pinned to one GPU, run with `-u`/`PYTHONUNBUFFERED` so logs stream live.

In [ ]:
import os, sys, subprocess, time, torch

n_gpus = torch.cuda.device_count()
nshards = max(1, n_gpus)
RUN_SHARD = f"{MODELING_DIR}/run_shard.py"
pathlib.Path("results").mkdir(exist_ok=True)

procs = []
for i in range(nshards):
    env = dict(os.environ); env["CUDA_VISIBLE_DEVICES"] = str(i)
    env["PYTHONUNBUFFERED"] = "1"
    logf = f"shard{i}.log"; fh = open(logf, "w")
    cmd = [sys.executable, "-u", RUN_SHARD, "--runlist", "runlist.json", "--shard", str(i),
           "--nshards", str(nshards), "--data-dir", DATA_DIR_NSE, "--out", f"results/shard{i}.csv",
           "--ckpt-dir", "checkpoints/tierc", "--area", "tier_c", "--epochs", "20"]
    procs.append([subprocess.Popen(cmd, env=env, stdout=fh, stderr=subprocess.STDOUT), fh, logf, 0])
    print(f"launched shard {i} on CUDA_VISIBLE_DEVICES={i}")

while any(p[0].poll() is None for p in procs):
    for p in procs:
        with open(p[2]) as r:
            r.seek(p[3]); chunk = r.read(); p[3] = r.tell()
        if chunk:
            print(chunk, end="")
    time.sleep(10)
for p in procs:                                        # final flush
    p[1].close()
    with open(p[2]) as r:
        r.seek(p[3]); print(r.read(), end="")
print("\nexit codes:", [p[0].poll() for p in procs])

## 7. Merge shard CSVs + push to S3

In [ ]:
import glob, pandas as pd
from nbenv import s3_client, s3_put_area
keys = ["model", "symbol", "feature_set", "horizon", "label_scheme", "seed"]
frames = [pd.read_csv(MERGED)] if MERGED.exists() else []
frames += [pd.read_csv(f) for f in glob.glob("results/shard*.csv")]
alldf = pd.concat(frames, ignore_index=True).drop_duplicates(subset=keys, keep="last") if frames else pd.DataFrame()
alldf.to_csv(MERGED, index=False)
s3_put_area(s3_client(), MERGED, AREA)
print(f"merged {len(alldf)} rows -> {MERGED}  (s3: experiments/{AREA}/results/)")
alldf.sort_values(keys)

## 8. Multi-seed aggregation — mean ± std and 95% CI across seeds
For each (model, feature_set, symbol, horizon) we summarise weighted/macro-$F_1$ over seeds. The 95% CI is a
Student-t interval on the seed mean (small-$n$ honest)..

In [ ]:
import numpy as np, pandas as pd
from scipy import stats as ss
res = pd.read_csv(MERGED)

def cfg_label(r): return f"{r.model}/{r.feature_set}"

def agg(metric):
    rows = []
    g = res.groupby(["model", "feature_set", "symbol", "horizon"])[metric]
    for (model, fs, sym, h), s in g:
        v = s.values.astype(float); n = len(v)
        mean = v.mean(); sd = v.std(ddof=1) if n > 1 else 0.0
        if n > 1:
            lo, hi = ss.t.interval(0.95, n - 1, loc=mean, scale=sd / np.sqrt(n))
        else:
            lo = hi = mean
        rows.append({"config": f"{model}/{fs}", "symbol": sym, "horizon": h, "n": n,
                     "mean": round(mean, 4), "std": round(sd, 4),
                     "ci95": f"[{lo:.4f}, {hi:.4f}]"})
    return pd.DataFrame(rows).sort_values(["symbol", "horizon", "config"])

print("=== weighted-F1 (mean +/- std, 95% CI over seeds) ===")
wf = agg("test_weighted_f1"); print(wf.to_string(index=False))
print("\n=== macro-F1 ===")
mf = agg("test_macro_f1"); print(mf.to_string(index=False))

# persist the summary
wf.to_csv("results/tier_c_summary_wf1.csv", index=False)
mf.to_csv("results/tier_c_summary_mf1.csv", index=False)
from nbenv import s3_client, s3_put_area
s3_put_area(s3_client(), "results/tier_c_summary_wf1.csv", AREA)
s3_put_area(s3_client(), "results/tier_c_summary_mf1.csv", AREA)

## 9. Significance tests across seeds (paired)
For each model pair we pair observations by (seed, symbol, horizon) and run a paired t-test + Wilcoxon
signed-rank on the weighted-$F_1$ difference. This answers "is A reliably better than B across the board?"
with the seeds as repeated measures. (Small $n$ -> we report the mean Δ, both p-values, and win-rate.)

In [ ]:
import numpy as np, pandas as pd
from scipy import stats as ss
res = pd.read_csv(MERGED)

def series(model, fs):
    s = res[(res.model == model) & (res.feature_set == fs)]
    return s.set_index(["seed", "symbol", "horizon"])["test_weighted_f1"]

PAIRS = [
    ("ConvMamba vs Mamba (Tier-B)",       ("convmambalob", "all"),   ("mambalob", "all")),
    ("Mamba all vs base40 (Tier-A)",      ("mambalob", "all"),       ("mambalob", "base40")),
    ("Mamba vs TLOB",                     ("mambalob", "all"),       ("tlob", "all")),
    ("ConvMamba vs TLOB",                 ("convmambalob", "all"),   ("tlob", "all")),
]
out = []
for name, (ma, fa), (mb, fb) in PAIRS:
    a, b = series(ma, fa), series(mb, fb)
    idx = a.index.intersection(b.index)
    da, db = a.loc[idx].values, b.loc[idx].values
    diff = da - db
    if len(diff) < 2:
        out.append({"comparison": name, "n_pairs": len(diff), "mean_delta": np.nan}); continue
    t_p = ss.ttest_rel(da, db).pvalue
    try:
        w_p = ss.wilcoxon(da, db).pvalue
    except ValueError:
        w_p = np.nan
    out.append({"comparison": name, "n_pairs": len(diff),
                "mean_delta_wF1": round(float(diff.mean()), 4),
                "wins_A": int((diff > 0).sum()),
                "ttest_p": round(float(t_p), 4),
                "wilcoxon_p": round(float(w_p), 4) if w_p == w_p else None})
sig = pd.DataFrame(out)
print(sig.to_string(index=False))
print("\n(positive mean_delta => first model better; p<0.05 => difference unlikely by chance)")
sig.to_csv("results/tier_c_significance.csv", index=False)
from nbenv import s3_client, s3_put_area
s3_put_area(s3_client(), "results/tier_c_significance.csv", AREA)

## 10. Per-sample paired bootstrap + McNemar (strongest evidence)
Across-seed tests have few points. Here we take a representative cell (**NIFTY, k=100, seed 0**), reload the
saved checkpoints from S3, recompute test predictions, and compare models *on the same test instances*:
a paired bootstrap on the weighted-$F_1$ difference (resampling test rows) and McNemar's test on per-sample
correctness. These have thousands of points, so they detect smaller real differences.

In [ ]:
import numpy as np, pathlib, torch
from scipy import stats as ss
from sklearn.metrics import f1_score
from nse_dataset import load_nse
from models import build_model
from train import load_checkpoint, DEVICE
from runner import collect_preds, spec_tag
from nbenv import s3_client, s3_pull_area

EVAL_SYMBOL, EVAL_H, EVAL_SEED = "NIFTY", 100, 0
_cache = {}
def _testset(symbol, horizon, fs):
    k = (symbol, horizon, fs)
    if k not in _cache:
        tr, vl, te = load_nse(symbol=symbol, horizon=horizon, seq_len=100,
                              feature_set=fs, label_scheme="A", data_dir=DATA_DIR_NSE)
        _cache[k] = (tr, te)
    return _cache[k]

def preds_for(model, fs):
    tr, te = _testset(EVAL_SYMBOL, EVAL_H, fs)
    n_features = tr[0][0].shape[-1]
    net = build_model(model, seq_len=100, n_features=n_features)
    tag = spec_tag({"model": model, "symbol": EVAL_SYMBOL, "feature_set": fs,
                    "horizon": EVAL_H, "label_scheme": "A", "seed": EVAL_SEED})
    ckpt = pathlib.Path(f"checkpoints/tierc/{tag}.pt")
    if not ckpt.exists():
        s3_pull_area(s3_client(), ckpt, AREA)
    net = load_checkpoint(net, str(ckpt))
    yp, yt = collect_preds(net, te)
    return yt, yp

def paired_bootstrap_wf1(yt, pa, pb, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed); n = len(yt)
    d0 = f1_score(yt, pa, average="weighted", zero_division=0) - \
         f1_score(yt, pb, average="weighted", zero_division=0)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        diffs[i] = f1_score(yt[idx], pa[idx], average="weighted", zero_division=0) - \
                   f1_score(yt[idx], pb[idx], average="weighted", zero_division=0)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return d0, lo, hi, float((diffs > 0).mean())

def mcnemar(yt, pa, pb):
    ca, cb = (pa == yt), (pb == yt)
    b = int((ca & ~cb).sum()); c = int((~ca & cb).sum())     # discordant
    # exact McNemar via binomial on discordant pairs
    p = ss.binomtest(min(b, c), b + c, 0.5).pvalue if (b + c) else 1.0
    return b, c, p

PAIRS = [("convmambalob", "all", "mambalob", "all"),
         ("mambalob", "all", "mambalob", "base40"),
         ("mambalob", "all", "tlob", "all")]
print(f"Per-sample comparison at {EVAL_SYMBOL} k={EVAL_H} seed={EVAL_SEED}\n" + "=" * 60)
for ma, fa, mb, fb in PAIRS:
    yt_a, pa = preds_for(ma, fa)
    yt_b, pb = preds_for(mb, fb)
    assert np.array_equal(yt_a, yt_b), "label mismatch -> test sets differ"
    yt = yt_a
    d0, lo, hi, frac = paired_bootstrap_wf1(yt, pa, pb)
    b, c, mp = mcnemar(yt, pa, pb)
    sig = "SIGNIF" if (lo > 0 or hi < 0) else "n.s."
    print(f"{ma}/{fa}  vs  {mb}/{fb}  (n={len(yt)})")
    print(f"   ΔwF1={d0:+.4f}  95%CI=[{lo:+.4f},{hi:+.4f}]  P(Δ>0)={frac:.2f}  [{sig}]")
    print(f"   McNemar: A-only-correct={b}, B-only-correct={c}, p={mp:.4g}\n")

## 11. Probability calibration (temperature scaling)
A classifier can be accurate but over-confident. We fit a single temperature $T$ on the validation logits
(minimising NLL) and report **Expected Calibration Error (ECE)** before/after on the test set, for the
proposed model. Lower ECE => probabilities are more trustworthy for the cost-aware backtest.

In [ ]:
import numpy as np, torch, torch.nn.functional as F
from nse_dataset import load_nse
from models import build_model
from train import load_checkpoint, DEVICE
from fi2010_dataset import make_loader
from runner import spec_tag
from nbenv import s3_client, s3_pull_area
import pathlib

CAL_MODEL, CAL_FS, CAL_SYM, CAL_H, CAL_SEED = "convmambalob", "all", "NIFTY", 100, 0

def logits_labels(net, ds):
    net = net.to(DEVICE).eval(); L, Y = [], []
    with torch.no_grad():
        for X, y in make_loader(ds, batch_size=256, shuffle=False):
            L.append(net(X.to(DEVICE)).cpu()); Y.append(y)
    return torch.cat(L), torch.cat(Y)

def ece(probs, labels, n_bins=15):
    conf, pred = probs.max(1)
    acc = (pred == labels).float()
    bins = torch.linspace(0, 1, n_bins + 1)
    e = 0.0
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i + 1])
        if m.any():
            e += m.float().mean() * (acc[m].mean() - conf[m].mean()).abs()
    return float(e)

tr, vl, te = load_nse(symbol=CAL_SYM, horizon=CAL_H, seq_len=100, feature_set=CAL_FS,
                      label_scheme="A", data_dir=DATA_DIR_NSE)
net = build_model(CAL_MODEL, seq_len=100, n_features=tr[0][0].shape[-1])
tag = spec_tag({"model": CAL_MODEL, "symbol": CAL_SYM, "feature_set": CAL_FS,
                "horizon": CAL_H, "label_scheme": "A", "seed": CAL_SEED})
ckpt = pathlib.Path(f"checkpoints/tierc/{tag}.pt")
if not ckpt.exists():
    s3_pull_area(s3_client(), ckpt, AREA)
net = load_checkpoint(net, str(ckpt))

val_logits, val_y = logits_labels(net, vl)
test_logits, test_y = logits_labels(net, te)

# fit temperature on validation NLL
T = torch.nn.Parameter(torch.ones(1))
opt = torch.optim.LBFGS([T], lr=0.01, max_iter=100)
def _closure():
    opt.zero_grad()
    loss = F.cross_entropy(val_logits / T.clamp(min=1e-2), val_y)
    loss.backward(); return loss
opt.step(_closure)
Tval = float(T.detach().clamp(min=1e-2))

ece_before = ece(F.softmax(test_logits, 1), test_y)
ece_after = ece(F.softmax(test_logits / Tval, 1), test_y)
print(f"{CAL_MODEL}/{CAL_FS} {CAL_SYM} k={CAL_H} seed={CAL_SEED}")
print(f"  fitted temperature T = {Tval:.3f}  ({'over' if Tval>1 else 'under'}-confident before)")
print(f"  ECE before = {ece_before:.4f}")
print(f"  ECE after  = {ece_after:.4f}   ({100*(ece_before-ece_after)/max(ece_before,1e-9):+.1f}% change)")

## 12. Tier-C summary

Read off, for the report:
- **Tier-A claim** (microstructure features help): see the `Mamba all vs base40` row in §9 (across-seed) and §10
  (per-sample) — significant if CI excludes 0 / p<0.05.
- **Tier-B claim** (ConvMambaLOB > base): `ConvMamba vs Mamba` rows.
- **Gap to TLOB**: `* vs TLOB` rows quantify it with uncertainty (expected: TLOB ahead, but report the CI).
- **Calibration**: temperature + ECE improvement (§11) — needed before trusting predicted probabilities in the
  cost-aware backtest.

All summary CSVs (`tier_c_summary_*`, `tier_c_significance`) are on S3 under `experiments/tier_c/results/`.
